# Welcome to your new notebook
# Type here in the cell editor to add code!


In [3]:
import pyspark.pandas as pd
from pyspark.sql.functions import monotonically_increasing_id,col, when,make_date,lit

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 5, Finished, Available, Finished)

In [4]:
df = spark.read.table("silver_actual")


StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 6, Finished, Available, Finished)

In [5]:
df.show(5,truncate=False)

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 7, Finished, Available, Finished)

+----+-------+-----+----------+---------------------------------------------------+--------------+------------+-------+-----------+------------+------------+--------+---------+-----------+--------+-----------+----------------+-----------+------------+-----------------+---------------------+------------------------+-----------------------------+------+-----------+----+---------+-----+-----+-----------------+------+------------+------------+-----+---------------+------------+----------+
|year|quarter|month|sku_code  |sku_description                                    |category      |sub_category|product|sub_product|cluster_head|channel     |location|volume_mt|gross_sales|discount|trade_spend|total_t_and_disc|net_revenue|raw_material|packging_material|industrial_fixed_cost|industrial_variable_cost|total_fixed_and_variable_cost|cogs  |goss_profit|gp% |marketing|sandd|ganda|other_inc_and_exp|ebitda|depriciation|one_off_item|tax  |interest_income|interest_exp|net_profit|
+----+-------+-----+

In [2]:
# Read Silver table
silver_actual = spark.read.table("silver_actual")

StatementMeta(, 71549568-9ec9-4893-b214-6a20b8167060, 4, Finished, Available, Finished)

In [6]:
from pyspark.sql.functions import monotonically_increasing_id



# Create Product Dimension
dim_product = (
    silver_actual
        .select(
            "sku_code",
            "sku_description",
            "category",
            "sub_category",
            "product",
            "sub_product"
        )
        .dropDuplicates()
        .withColumn("product_key", monotonically_increasing_id())
)



StatementMeta(, 71549568-9ec9-4893-b214-6a20b8167060, 8, Finished, Available, Finished)

In [7]:
dim_product.count()

StatementMeta(, 71549568-9ec9-4893-b214-6a20b8167060, 9, Finished, Available, Finished)

4207

In [9]:
dim_product.show()

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 11, Finished, Available, Finished)

+----------+--------------------+--------------+-----------------+-------------+----------------+-----------+
|  sku_code|     sku_description|      category|     sub_category|      product|     sub_product|product_key|
+----------+--------------------+--------------+-----------------+-------------+----------------+-----------+
|4040484088|CG 400g Hommos Ta...|    Fresh Fare|             Foul|   Foul Plain|      Foul Plain|          0|
|4001971961|XTREME FLAMING CH...|  Protein pack|           Frozen|      Chicken|          Strips|          1|
|4083334089|Chips Lion Vinega...|Crunch & Munch|         Biscuits|        Wafer|           Wafer|          2|
|4082002004|Biscuit OG Coated...|Crunch & Munch|         Biscuits|        Wafer|           Wafer|          3|
|4083018048|Windows CHEESE 6 ...|Crunch & Munch|         Biscuits|        Wafer|           Wafer|          4|
|4036010897|Gobber Whole Stra...|Frosty Veggies|Frozen Vegetables|   Sweet Corn|Sweet Corn Seeds|          5|
|403699909

In [8]:
# Reordering
dim_product = dim_product.select(
    "product_key",
    "sku_code",
    "sku_description",
    "category",
    "sub_category",
    "product",
    "sub_product"
)


StatementMeta(, 71549568-9ec9-4893-b214-6a20b8167060, 10, Finished, Available, Finished)

In [9]:
dim_product.show(5)

StatementMeta(, 71549568-9ec9-4893-b214-6a20b8167060, 11, Finished, Available, Finished)

+-----------+----------+--------------------+--------------+------------+----------+-----------+
|product_key|  sku_code|     sku_description|      category|sub_category|   product|sub_product|
+-----------+----------+--------------------+--------------+------------+----------+-----------+
|          0|4040484088|CG 400g Hommos Ta...|    Fresh Fare|        Foul|Foul Plain| Foul Plain|
|          1|4001971961|XTREME FLAMING CH...|  Protein pack|      Frozen|   Chicken|     Strips|
|          2|4083334089|Chips Lion Vinega...|Crunch & Munch|    Biscuits|     Wafer|      Wafer|
|          3|4082002004|Biscuit OG Coated...|Crunch & Munch|    Biscuits|     Wafer|      Wafer|
|          4|4083018048|Windows CHEESE 6 ...|Crunch & Munch|    Biscuits|     Wafer|      Wafer|
+-----------+----------+--------------------+--------------+------------+----------+-----------+
only showing top 5 rows



In [10]:
dim_product.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_product")


StatementMeta(, 71549568-9ec9-4893-b214-6a20b8167060, 12, Finished, Available, Finished)

In [13]:
#Creating dim_cluster
dim_cluster=silver_actual\
            .select('cluster_head').dropDuplicates()\
            .withColumn("cluster_id",monotonically_increasing_id())

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 15, Finished, Available, Finished)

In [14]:
dim_cluster.show()

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 16, Finished, Available, Finished)

+------------+----------+
|cluster_head|cluster_id|
+------------+----------+
|      Dhiraj|         0|
|        Umar|         1|
|      Kasfur|         2|
|     Harshal|         3|
|        Anas|         4|
|       Bilal|         5|
|     Shreyas|         6|
|    Priyanka|         7|
+------------+----------+



In [15]:
dim_cluster=dim_cluster.select('cluster_id','cluster_head')

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 17, Finished, Available, Finished)

In [16]:
dim_cluster.show()

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 18, Finished, Available, Finished)

+----------+------------+
|cluster_id|cluster_head|
+----------+------------+
|         0|      Dhiraj|
|         1|        Umar|
|         2|      Kasfur|
|         3|     Harshal|
|         4|        Anas|
|         5|       Bilal|
|         6|     Shreyas|
|         7|    Priyanka|
+----------+------------+



In [17]:
dim_cluster.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_cluster")


StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 19, Finished, Available, Finished)

In [18]:
#Creating Channel_dim
channel_dim=silver_actual\
.select('channel').dropDuplicates()\
.withColumn("channel_id",monotonically_increasing_id())

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 20, Finished, Available, Finished)

In [19]:
dim_channel=channel_dim.select('channel_id','channel')

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 21, Finished, Available, Finished)

In [20]:
dim_channel.show()

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 22, Finished, Available, Finished)

+----------+----------------+
|channel_id|         channel|
+----------+----------------+
|         0|Culinary Service|
|         1|      Bulk Sales|
|         2|    Online Sales|
|         3|  External Sales|
|         4|          others|
|         5|    Direct Sales|
|         6|  Domestic Sales|
|         7|     Distributor|
|         8|       Mfg Sales|
+----------+----------------+



In [21]:
dim_channel.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_dim_channel")

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 23, Finished, Available, Finished)

In [22]:
# Dim location
dim_location=silver_actual\
.select('location')\
.dropDuplicates()\
.withColumn("location_id",monotonically_increasing_id())



StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 24, Finished, Available, Finished)

In [23]:
dim_location=dim_location.select('location_id','location')

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 25, Finished, Available, Finished)

In [24]:
dim_location.show()

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 26, Finished, Available, Finished)

+-----------+----------+
|location_id|  location|
+-----------+----------+
|          0| Karnataka|
|          1|Tamil Nadu|
|          2|       Guj|
|          3|        Up|
|          4|       Raj|
|          5|  Calcutta|
|          6|       Mah|
+-----------+----------+



In [25]:
dim_location.write\
    .format('delta').mode("overwrite")\
    .option("overwriteSchema", "true")\
    .saveAsTable("gold_dim_location")

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 27, Finished, Available, Finished)

In [26]:
#Dim date

dim_date=silver_actual.select("Year","Quarter","Month")\
.dropDuplicates()\
.withColumn("date_id",monotonically_increasing_id())

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 28, Finished, Available, Finished)

In [27]:
dim_date=dim_date.select("date_id","Month","Quarter","Year")

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 29, Finished, Available, Finished)

In [28]:
dim_date.show()

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 30, Finished, Available, Finished)

+-------+-----+-------+----+
|date_id|Month|Quarter|Year|
+-------+-----+-------+----+
|      0|  mar|     Q1|2024|
|      1|  oct|     Q4|2024|
|      2|  mar|     Q1|2021|
|      3|  jul|     Q3|2022|
|      4|  may|     Q2|2022|
|      5|  apr|     Q2|2021|
|      6|  jan|     Q1|2020|
|      7|  oct|     Q4|2021|
|      8|  sep|     Q3|2023|
|      9|  may|     Q2|2020|
|     10|  aug|     Q3|2023|
|     11|  aug|     Q3|2024|
|     12|  nov|     Q4|2023|
|     13|  nov|     Q4|2024|
|     14|  jun|     Q2|2022|
|     15|  jan|     Q1|2023|
|     16|  jun|     Q2|2020|
|     17|  feb|     Q1|2023|
|     18|  jul|     Q3|2024|
|     19|  jul|     Q3|2020|
+-------+-----+-------+----+
only showing top 20 rows



In [29]:
dim_date.count()

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 31, Finished, Available, Finished)

60

In [30]:
dim_date=dim_date.select("Date_id","Year","Quarter","Month")\
        .withColumn("month_no",
            when(col("Month").isin("jan", "Jan", "JAN"), 1)
            .when(col("Month").isin("feb", "Feb", "FEB"), 2)
            .when(col("Month").isin("mar", "Mar", "MAR"), 3)
            .when(col("Month").isin("apr", "Apr", "APR"), 4)
            .when(col("Month").isin("may", "May", "MAY"), 5)
            .when(col("Month").isin("jun", "Jun", "JUN"), 6)
            .when(col("Month").isin("jul", "Jul", "JUL"), 7)
            .when(col("Month").isin("aug", "Aug", "AUG"), 8)
            .when(col("Month").isin("sep", "Sep", "SEP"), 9)
            .when(col("Month").isin("oct", "Oct", "OCT"), 10)
            .when(col("Month").isin("nov", "Nov", "NOV"), 11)
            .when(col("Month").isin("dec", "Dec", "DEC"), 12)
        )

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 32, Finished, Available, Finished)

In [31]:
dim_date.show()

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 33, Finished, Available, Finished)

+-------+----+-------+-----+--------+
|Date_id|Year|Quarter|Month|month_no|
+-------+----+-------+-----+--------+
|      0|2024|     Q1|  mar|       3|
|      1|2024|     Q4|  oct|      10|
|      2|2021|     Q1|  mar|       3|
|      3|2022|     Q3|  jul|       7|
|      4|2022|     Q2|  may|       5|
|      5|2021|     Q2|  apr|       4|
|      6|2020|     Q1|  jan|       1|
|      7|2021|     Q4|  oct|      10|
|      8|2023|     Q3|  sep|       9|
|      9|2020|     Q2|  may|       5|
|     10|2023|     Q3|  aug|       8|
|     11|2024|     Q3|  aug|       8|
|     12|2023|     Q4|  nov|      11|
|     13|2024|     Q4|  nov|      11|
|     14|2022|     Q2|  jun|       6|
|     15|2023|     Q1|  jan|       1|
|     16|2020|     Q2|  jun|       6|
|     17|2023|     Q1|  feb|       2|
|     18|2024|     Q3|  jul|       7|
|     19|2020|     Q3|  jul|       7|
+-------+----+-------+-----+--------+
only showing top 20 rows



In [32]:
dim_date=dim_date\
.withColumn("date",make_date(
                col("Year"),
                col("month_no"),
                lit(1)   # day = 1 (start of month)
            ))

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 34, Finished, Available, Finished)

In [33]:
dim_date.show()

StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 35, Finished, Available, Finished)

+-------+----+-------+-----+--------+----------+
|Date_id|Year|Quarter|Month|month_no|      date|
+-------+----+-------+-----+--------+----------+
|      0|2024|     Q1|  mar|       3|2024-03-01|
|      1|2024|     Q4|  oct|      10|2024-10-01|
|      2|2021|     Q1|  mar|       3|2021-03-01|
|      3|2022|     Q3|  jul|       7|2022-07-01|
|      4|2022|     Q2|  may|       5|2022-05-01|
|      5|2021|     Q2|  apr|       4|2021-04-01|
|      6|2020|     Q1|  jan|       1|2020-01-01|
|      7|2021|     Q4|  oct|      10|2021-10-01|
|      8|2023|     Q3|  sep|       9|2023-09-01|
|      9|2020|     Q2|  may|       5|2020-05-01|
|     10|2023|     Q3|  aug|       8|2023-08-01|
|     11|2024|     Q3|  aug|       8|2024-08-01|
|     12|2023|     Q4|  nov|      11|2023-11-01|
|     13|2024|     Q4|  nov|      11|2024-11-01|
|     14|2022|     Q2|  jun|       6|2022-06-01|
|     15|2023|     Q1|  jan|       1|2023-01-01|
|     16|2020|     Q2|  jun|       6|2020-06-01|
|     17|2023|     Q

In [34]:
dim_date.write.format('delta')\
.mode("overwrite")\
.option("overwriteSchema", "true")\
.saveAsTable("gold_dim_date")


StatementMeta(, 20a3bbed-ffc6-4a78-a5ee-46fc5dd555c3, 36, Finished, Available, Finished)